In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain_neo4j import Neo4jGraph
from configuration.config import *


graph = Neo4jGraph(
    url=NEO4j_CONFIG["uri"],
    username=NEO4j_CONFIG["auth"][0],
    password=NEO4j_CONFIG["auth"][1],

)

In [3]:
print(graph.schema)

Node properties:
Category1 {id: INTEGER, name: STRING}
Category2 {id: INTEGER, name: STRING}
Category3 {id: INTEGER, name: STRING}
BaseAttrName {id: INTEGER, name: STRING}
BaseAttrValue {id: INTEGER, name: STRING}
SPU {id: INTEGER, name: STRING}
SKU {id: INTEGER, name: STRING}
Trademark {id: INTEGER, name: STRING}
SaleAttrName {id: INTEGER, name: STRING}
SaleAttrValue {id: INTEGER, name: STRING}
Tag {id: STRING, name: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:Category3)-[:Have]->(:BaseAttrName)
(:BaseAttrName)-[:Have]->(:BaseAttrValue)
(:SPU)-[:Have]->(:Tag)
(:SPU)-[:Have]->(:SaleAttrName)
(:SPU)-[:Belong]->(:Trademark)
(:SPU)-[:Belong]->(:Category3)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:SaleAttrName)-[:Have]->(:SaleAttrValue)


In [4]:
cypher = "Match (n) Return n LIMIT 5"
graph.query(cypher)

[{'n': {'name': '手机饰品', 'id': 83}},
 {'n': {'name': '拍照配件', 'id': 84}},
 {'n': {'name': '手机支架', 'id': 85}},
 {'n': {'name': '平板电视', 'id': 86}},
 {'n': {'name': '空调', 'id': 87}}]

In [5]:
from langchain_deepseek import ChatDeepSeek

llm =ChatDeepSeek(
    model = "deepseek-chat",
    api_key = DEEPSEEK_API_KEY,
    temperature=1
)

In [7]:
from langchain_neo4j import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True,allow_dangerous_requests=True)

In [8]:
result = chain.invoke({"query":"华为有哪些产品"})




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '华为'})-[:Belong]-(:SPU)-[:Belong]-(:Category3)
MATCH (s:SKU)-[:Belong]->(:SPU)-[:Belong]->(t)
RETURN DISTINCT s.name AS productName
Full Context:
[{'productName': '华为智慧屏V55i-J 55英寸 HEGE-550B 4K全面屏智能电视机 多方视频通话 AI升降摄像头 银钻灰 京品家电'}, {'productName': '华为智慧屏V65i 65英寸 HEGE-560B 4K全面屏智能电视机 多方视频通话 AI升降摄像头 4GB+32GB 星际黑'}, {'productName': '华为Mate 40 pro 5G全网通麒麟9000芯片鸿蒙国货NFC双卡游戏学生 黑色 8+128 送适用100w充电器'}, {'productName': '华为HUAWEI二手笔记本MateBook13触屏2K全面屏 便携二手笔记本电脑 Magic R5-2500-8G-256G-高分屏 95成新'}, {'productName': 'HAWEIAI Book【补贴30%】2025英特尔14 Pro满血独显笔记本电脑轻薄本高端高性能金属游戏设计大学生【2.5K全面屏+抗蓝光+指纹】 32G内存+2TB超速硬盘'}]

> Finished chain.


In [9]:
result

{'query': '华为有哪些产品',
 'result': '华为的产品包括：华为智慧屏V55i-J 55英寸 4K全面屏智能电视机、华为智慧屏V65i 65英寸 4K全面屏智能电视机、华为Mate 40 Pro 5G全网通智能手机、华为MateBook 13触屏二手笔记本电脑，以及一款HAWEIAI Book 2025英特尔14 Pro笔记本电脑。'}